# Credit Card Default Prediction — v2 (Full Improvement Pipeline)

**Improvements over v1:**
1. **Feature Engineering** — income-per-person, age-at-first-job, tenure ratio, month bins
2. **Recall-Oriented Scoring** — optimise for recall @ precision ≥ 10% (credit risk priority)
3. **Hyperparameter Tuning** — `RandomizedSearchCV` on best candidates (Balanced RF + XGBoost_weighted)
4. **Smart Threshold Selection** — find threshold that maximises recall subject to precision floor, on a validation set (not test set)
5. **Stacking Ensemble** — Balanced RF + XGBoost_weighted as base learners, LogReg meta-learner

**Flow:**
```
Imports → Data → Feature Engineering → Split → Preprocessor → Imbalance Check
→ Baseline Eval → Hyperparameter Tuning → Smart Threshold → Stacking → Final Leaderboard
```

## Step 1 — Imports

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_val_score, RandomizedSearchCV
)
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_score, recall_score, f1_score,
    roc_auc_score, make_scorer, precision_recall_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    StackingClassifier
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.calibration import CalibratedClassifierCV

from xgboost import XGBClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.ensemble import BalancedRandomForestClassifier

from scipy.stats import randint, uniform

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42

## Step 2 — Data Loading & Merging

In [2]:
csv1 = "../archive/Credit_Card_Dataset_2025_Sept_1.csv"
csv2 = "../archive/Credit_Card_Dataset_2025_Sept_2.csv"

df1 = pd.read_csv(csv1)
df2 = pd.read_csv(csv2)

df = pd.merge(df1, df2, left_on='ID', right_on='User', how='inner')
df = df.drop(columns=[df.columns[0], "ID", "User", "FLAG_MOBIL"])
df = df[df["AGE"] < 100]

print(f"Dataset shape: {df.shape}")
print(f"Target distribution:\n{df['TARGET'].value_counts(normalize=True).round(4)}")
df.head()

Dataset shape: (25129, 17)
Target distribution:
TARGET
0    0.9832
1    0.0168
Name: proportion, dtype: float64


,GENDER,CAR,REALITY,NO_OF_CHILD,FAMILY_TYPE,HOUSE_TYPE,WORK_PHONE,PHONE,E_MAIL,FAMILY SIZE,BEGIN_MONTH,AGE,YEARS_EMPLOYED,TARGET,INCOME,INCOME_TYPE,EDUCATION_TYPE
0,M,Y,Y,0,Married,House / apartment,0,0,0,2.0,29,59,3.0,0,112500.0,Working,Secondary / secondary special
1,F,N,Y,0,Single / not married,House / apartment,0,1,1,1.0,4,52,8.0,0,270000.0,Commercial associate,Secondary / secondary special
2,F,N,Y,0,Single / not married,House / apartment,0,1,1,1.0,26,52,8.0,0,270000.0,Commercial associate,Secondary / secondary special
3,F,N,Y,0,Single / not married,House / apartment,0,1,1,1.0,26,52,8.0,0,270000.0,Commercial associate,Secondary / secondary special
4,F,N,Y,0,Single / not married,House / apartment,0,1,1,1.0,38,52,8.0,0,270000.0,Commercial associate,Secondary / secondary special


## Step 3 — Feature Engineering (Improvement #1)

Raw columns alone leave a lot on the table. We create domain-informed features:

| Feature | Formula | Rationale |
|---|---|---|
| `INCOME_PER_PERSON` | INCOME / FAMILY SIZE | Financial pressure per household member |
| `AGE_AT_FIRST_JOB` | AGE − YEARS_EMPLOYED | Late starters may be riskier |
| `TENURE_RATIO` | YEARS_EMPLOYED / AGE | Stability relative to age |
| `CHILDREN_RATIO` | NO_OF_CHILD / FAMILY SIZE | Dependent-load proxy |
| `BEGIN_MONTH_BIN` | Binned into quarters | Captures seasonal credit cycles |
| `HIGH_INCOME` | INCOME > 75th percentile | Binary affluence flag |

In [3]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Guard against division by zero
    family_size_safe = df["FAMILY SIZE"].replace(0, np.nan)
    age_safe         = df["AGE"].replace(0, np.nan)

    df["INCOME_PER_PERSON"] = df["INCOME"] / family_size_safe
    df["AGE_AT_FIRST_JOB"]  = df["AGE"] - df["YEARS_EMPLOYED"].clip(lower=0)
    df["TENURE_RATIO"]      = df["YEARS_EMPLOYED"].clip(lower=0) / age_safe
    df["CHILDREN_RATIO"]    = df["NO_OF_CHILD"] / family_size_safe

    # Bin BEGIN_MONTH into quarters (months are negative in this dataset convention)
    df["BEGIN_MONTH_BIN"] = pd.cut(
        df["BEGIN_MONTH"].abs(),
        bins=[0, 12, 24, 36, np.inf],
        labels=["0-1yr", "1-2yr", "2-3yr", "3yr+"]
    ).astype(str)

    # High income flag (computed on whole df; will be recomputed safely per-split below)
    income_75 = df["INCOME"].quantile(0.75)
    df["HIGH_INCOME"] = (df["INCOME"] > income_75).astype(int)

    return df


df_fe = engineer_features(df)

new_num_cols = ["INCOME_PER_PERSON", "AGE_AT_FIRST_JOB", "TENURE_RATIO", "CHILDREN_RATIO", "HIGH_INCOME"]
new_cat_cols = ["BEGIN_MONTH_BIN"]

print(f"New numeric features : {new_num_cols}")
print(f"New categorical features: {new_cat_cols}")
df_fe[new_num_cols + new_cat_cols].describe()

New numeric features : ['INCOME_PER_PERSON', 'AGE_AT_FIRST_JOB', 'TENURE_RATIO', 'CHILDREN_RATIO', 'HIGH_INCOME']
New categorical features: ['BEGIN_MONTH_BIN']


,INCOME_PER_PERSON,AGE_AT_FIRST_JOB,TENURE_RATIO,CHILDREN_RATIO,HIGH_INCOME
count,25128.000000,25120.000000,25120.000000,25128.000000,25129.000000
mean,101970.107981,33.334395,0.174225,0.153082,0.240519
std,75498.784089,9.515177,0.135907,0.210183,0.427407
min,5625.000000,18.000000,0.000000,0.000000,0.000000
25%,54000.000000,26.000000,0.071429,0.000000,0.000000
50%,78750.000000,32.000000,0.140000,0.000000,0.000000
75%,126000.000000,40.000000,0.243902,0.333333,0.000000
max,900000.000000,65.000000,0.693548,2.000000,1.000000


## Step 4 — Feature Definition & Train/Test/Val Split

> We carve out a **validation set** (10% of train) used **only** for threshold selection — the test set stays completely untouched until the final evaluation.

In [4]:
num_cols = [
    "NO_OF_CHILD", "BEGIN_MONTH", "AGE", "YEARS_EMPLOYED", "INCOME", "FAMILY SIZE",
    "INCOME_PER_PERSON", "AGE_AT_FIRST_JOB", "TENURE_RATIO", "CHILDREN_RATIO", "HIGH_INCOME"
]
cat_cols = [
    "GENDER", "CAR", "REALITY", "FAMILY_TYPE", "HOUSE_TYPE",
    "WORK_PHONE", "PHONE", "E_MAIL", "INCOME_TYPE", "EDUCATION_TYPE",
    "BEGIN_MONTH_BIN"
]

X = df_fe[num_cols + cat_cols]
y = df_fe["TARGET"]

# Train+Val / Test split
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# Train / Val split (val used only for threshold tuning)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.125,   # 0.125 × 0.8 = 10% of total
    stratify=y_trainval,
    random_state=RANDOM_STATE
)

print(f"Train : {len(X_train):,}  |  Val : {len(X_val):,}  |  Test : {len(X_test):,}")
print(f"Train class balance:\n{y_train.value_counts(normalize=True).round(4)}")

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f"\nImbalance ratio: {scale_pos_weight:.2f}x  →  scale_pos_weight for XGBoost")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

Train : 17,590  |  Val : 2,513  |  Test : 5,026
Train class balance:
TARGET
0    0.9832
1    0.0168
Name: proportion, dtype: float64

Imbalance ratio: 58.43x  →  scale_pos_weight for XGBoost


## Step 5 — Preprocessing Pipeline

In [5]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols)
])

print("Preprocessor ready.")

Preprocessor ready.


## Step 6 — Recall-Oriented Scoring (Improvement #2)

In credit default prediction, **missing a defaulter (FN) is far more costly than a false alarm (FP)**.

We define a custom scorer: **maximise recall subject to precision ≥ 10%**.  
This scorer returns recall when precision is acceptable, and 0 otherwise — forcing the search to avoid precision collapse.

In [6]:
MIN_PRECISION = 0.10  # Minimum acceptable precision

def recall_at_min_precision(y_true, y_pred, min_precision=MIN_PRECISION):
    """Returns recall if precision >= min_precision, else 0."""
    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    return r if p >= min_precision else 0.0

recall_constrained = make_scorer(recall_at_min_precision, greater_is_better=True)

print(f"Custom scorer: recall @ precision >= {MIN_PRECISION:.0%}")
print("This scorer will be used in RandomizedSearchCV and threshold selection.")

Custom scorer: recall @ precision >= 10%
This scorer will be used in RandomizedSearchCV and threshold selection.


## Step 7 — Evaluation Utilities

In [7]:
def predict_with_threshold(model, X, threshold=0.5):
    y_probs = model.predict_proba(X)[:, 1]
    return (y_probs >= threshold).astype(int)


def find_best_threshold(model, X_val, y_val, min_precision=MIN_PRECISION):
    """
    Improvement #4: Find threshold that maximises recall on the VALIDATION set
    subject to precision >= min_precision. Uses the precision-recall curve so
    every possible threshold is evaluated — not just a hand-picked grid.
    """
    y_proba = model.predict_proba(X_val)[:, 1]
    precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba)

    best_threshold = 0.5
    best_recall    = 0.0

    for p, r, t in zip(precisions[:-1], recalls[:-1], thresholds):
        if p >= min_precision and r > best_recall:
            best_recall    = r
            best_threshold = t

    return best_threshold, best_recall


def evaluate_single(name, model, threshold, fitted=True):
    """Evaluate a fitted model on X_test at a given threshold."""
    if not fitted:
        model.fit(X_train, y_train)

    y_pred  = predict_with_threshold(model, X_test, threshold)
    y_proba = model.predict_proba(X_test)[:, 1]

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    return {
        "Model":      name,
        "Threshold":  threshold,
        "Precision":  precision_score(y_test, y_pred, zero_division=0),
        "Recall":     recall_score(y_test, y_pred, zero_division=0),
        "F1":         f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC":    roc_auc_score(y_test, y_proba),
        "TP": tp, "TN": tn, "FP": fp, "FN": fn
    }


def style_results(df, sort_by="Recall"):
    num_fmt = {
        c: "{:.4f}" for c in ["Precision","Recall","F1","ROC-AUC","Threshold",
                               "CV F1 Mean","CV F1 Std"]
        if c in df.columns
    }
    int_fmt = {c: "{:,}" for c in ["TP","TN","FP","FN"] if c in df.columns}
    fmt = {**num_fmt, **int_fmt}

    grad_cols_g  = [c for c in ["TP","TN","F1","Recall","ROC-AUC"] if c in df.columns]
    grad_cols_r  = [c for c in ["FP","FN"] if c in df.columns]

    styled = df.sort_values(sort_by, ascending=False).style
    if grad_cols_g: styled = styled.background_gradient(cmap='RdYlGn',   subset=grad_cols_g)
    if grad_cols_r: styled = styled.background_gradient(cmap='RdYlGn_r', subset=grad_cols_r)
    return styled.format(fmt)


print("Evaluation utilities ready.")

Evaluation utilities ready.


## Step 8 — Baseline Models (v1 Best Performers)

We re-run the two v1 winners — Balanced RF and XGBoost_weighted — as a reference point before tuning.

In [8]:
base_brf = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', BalancedRandomForestClassifier(
        n_estimators=100, random_state=RANDOM_STATE
    ))
])

base_xgb = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss', random_state=RANDOM_STATE
    ))
])

print("Fitting baseline candidates...")
base_brf.fit(X_train, y_train)
base_xgb.fit(X_train, y_train)

baseline_rows = [
    evaluate_single("Balanced RF (base)",      base_brf, threshold=0.5),
    evaluate_single("XGBoost_weighted (base)", base_xgb, threshold=0.5),
]
baseline_df = pd.DataFrame(baseline_rows)
print("\n=== BASELINE (pre-tuning, threshold=0.5) ===")
display(style_results(baseline_df))

Fitting baseline candidates...

=== BASELINE (pre-tuning, threshold=0.5) ===


,Model,Threshold,Precision,Recall,F1,ROC-AUC,TP,TN,FP,FN
0,Balanced RF (base),0.5000,0.0594,0.4762,0.1057,0.7523,40,"4,309",633,44
1,XGBoost_weighted (base),0.5000,0.1070,0.4524,0.1731,0.7703,38,"4,625",317,46


## Step 9 — Hyperparameter Tuning (Improvement #3)

`RandomizedSearchCV` over 40 iterations with 5-fold CV, scored by our **recall-constrained** scorer.  
Tuning on `X_train` only — val and test are never seen.

> ⚠️ This cell takes ~5–15 minutes depending on your hardware. Set `n_iter` lower to speed it up.

In [9]:
# ── Balanced Random Forest search space ────────────────────────────────────────
brf_param_dist = {
    'classifier__n_estimators':      randint(100, 500),
    'classifier__max_depth':         [None, 10, 20, 30],
    'classifier__min_samples_split': randint(2, 20),
    'classifier__min_samples_leaf':  randint(1, 10),
    'classifier__max_features':      ['sqrt', 'log2', 0.5],
    'classifier__sampling_strategy': ['auto', 0.5, 0.75],
}

brf_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', BalancedRandomForestClassifier(random_state=RANDOM_STATE))
])

brf_search = RandomizedSearchCV(
    brf_pipeline,
    param_distributions=brf_param_dist,
    n_iter=40,
    scoring=recall_constrained,
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=RANDOM_STATE
)

print("Tuning Balanced RF...")
brf_search.fit(X_train, y_train)
print(f"\nBest CV score (recall@prec≥{MIN_PRECISION:.0%}): {brf_search.best_score_:.4f}")
print(f"Best params: {brf_search.best_params_}")

tuned_brf = brf_search.best_estimator_

Tuning Balanced RF...
Fitting 5 folds for each of 40 candidates, totalling 200 fits

Best CV score (recall@prec≥10%): 0.1017
Best params: {'classifier__max_depth': 30, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 13, 'classifier__n_estimators': 413, 'classifier__sampling_strategy': 0.5}


In [10]:
# ── XGBoost search space ────────────────────────────────────────────────────────
xgb_param_dist = {
    'classifier__n_estimators':     randint(100, 600),
    'classifier__max_depth':        randint(3, 10),
    'classifier__learning_rate':    uniform(0.01, 0.2),
    'classifier__subsample':        uniform(0.6, 0.4),
    'classifier__colsample_bytree': uniform(0.6, 0.4),
    'classifier__min_child_weight': randint(1, 10),
    'classifier__gamma':            uniform(0, 0.5),
    'classifier__reg_alpha':        uniform(0, 1),
    'classifier__reg_lambda':       uniform(0.5, 2),
    'classifier__scale_pos_weight': [scale_pos_weight, scale_pos_weight * 1.5, scale_pos_weight * 2],
}

xgb_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        eval_metric='logloss', random_state=RANDOM_STATE
    ))
])

xgb_search = RandomizedSearchCV(
    xgb_pipeline,
    param_distributions=xgb_param_dist,
    n_iter=40,
    scoring=recall_constrained,
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=RANDOM_STATE
)

print("Tuning XGBoost...")
xgb_search.fit(X_train, y_train)
print(f"\nBest CV score (recall@prec≥{MIN_PRECISION:.0%}): {xgb_search.best_score_:.4f}")
print(f"Best params: {xgb_search.best_params_}")

tuned_xgb = xgb_search.best_estimator_

Tuning XGBoost...
Fitting 5 folds for each of 40 candidates, totalling 200 fits

Best CV score (recall@prec≥10%): 0.3853
Best params: {'classifier__colsample_bytree': np.float64(0.9208787923016158), 'classifier__gamma': np.float64(0.03727532183988541), 'classifier__learning_rate': np.float64(0.20737738732010347), 'classifier__max_depth': 3, 'classifier__min_child_weight': 8, 'classifier__n_estimators': 571, 'classifier__reg_alpha': np.float64(0.014079822715084456), 'classifier__reg_lambda': np.float64(0.8976848081776103), 'classifier__scale_pos_weight': np.float64(116.85135135135135), 'classifier__subsample': np.float64(0.9160702162124823)}


## Step 10 — Smart Threshold Selection (Improvement #4)

Instead of hand-picking a threshold, we sweep the **full precision-recall curve** on the **validation set** and pick the threshold that maximises recall while keeping precision ≥ 10%.  
The test set is still unseen at this point.

In [11]:
models_to_tune_threshold = {
    "Balanced RF (base)":    base_brf,
    "XGBoost_weighted (base)": base_xgb,
    "Balanced RF (tuned)":   tuned_brf,
    "XGBoost (tuned)":       tuned_xgb,
}

best_thresholds = {}

print(f"Finding optimal thresholds on validation set (precision >= {MIN_PRECISION:.0%})...\n")
for name, model in models_to_tune_threshold.items():
    t, r = find_best_threshold(model, X_val, y_val)
    best_thresholds[name] = t
    print(f"  {name:<35} → threshold={t:.3f}  (val recall={r:.4f})")

Finding optimal thresholds on validation set (precision >= 10%)...

  Balanced RF (base)                  → threshold=0.618  (val recall=0.2143)
  XGBoost_weighted (base)             → threshold=0.488  (val recall=0.3810)
  Balanced RF (tuned)                 → threshold=0.559  (val recall=0.2143)
  XGBoost (tuned)                     → threshold=0.354  (val recall=0.5238)


## Step 11 — Stacking Ensemble (Improvement #5)

Tuned Balanced RF and Tuned XGBoost as base learners.  
A Logistic Regression meta-learner learns how to combine their probability outputs.  
We use `passthrough=True` so the meta-learner also sees the original features.

In [12]:
# Extract the fitted estimators from tuned pipelines to use as base learners
# We wrap raw sklearn estimators (not ImbPipelines) for StackingClassifier
# The preprocessor is applied once at the top level

brf_params = {k.replace('classifier__', ''): v
              for k, v in brf_search.best_params_.items()}
xgb_params = {k.replace('classifier__', ''): v
              for k, v in xgb_search.best_params_.items()}

stacking_clf = StackingClassifier(
    estimators=[
        ('brf', BalancedRandomForestClassifier(**brf_params, random_state=RANDOM_STATE)),
        ('xgb', XGBClassifier(**xgb_params, eval_metric='logloss', random_state=RANDOM_STATE)),
    ],
    final_estimator=LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
    cv=cv,
    passthrough=False,  # meta-learner sees only base-learner predictions
    n_jobs=-1
)

stacking_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', stacking_clf)
])

print("Fitting stacking ensemble...")
stacking_pipeline.fit(X_train, y_train)
print("Done.")

# Find best threshold for stacker too
stack_threshold, stack_val_recall = find_best_threshold(stacking_pipeline, X_val, y_val)
best_thresholds["Stacking (BRF + XGB)"] = stack_threshold
print(f"Stacking threshold → {stack_threshold:.3f}  (val recall={stack_val_recall:.4f})")

Fitting stacking ensemble...
Done.
Stacking threshold → 0.691  (val recall=0.4286)


## Step 12 — Final Evaluation on Test Set

**First and only time the test set is used for all tuned models.**  
Each model is evaluated at its val-optimised threshold.

In [13]:
all_models_eval = {
    "Balanced RF (base)":      (base_brf,            best_thresholds["Balanced RF (base)"]),
    "XGBoost_weighted (base)": (base_xgb,            best_thresholds["XGBoost_weighted (base)"]),
    "Balanced RF (tuned)":     (tuned_brf,           best_thresholds["Balanced RF (tuned)"]),
    "XGBoost (tuned)":         (tuned_xgb,           best_thresholds["XGBoost (tuned)"]),
    "Stacking (BRF + XGB)":    (stacking_pipeline,   best_thresholds["Stacking (BRF + XGB)"]),
}

final_rows = []
for name, (model, threshold) in all_models_eval.items():
    row = evaluate_single(name, model, threshold=threshold, fitted=True)
    final_rows.append(row)
    print(f"{name:<35} | thresh={threshold:.3f} | "
          f"Recall={row['Recall']:.4f}  Prec={row['Precision']:.4f}  "
          f"F1={row['F1']:.4f}  AUC={row['ROC-AUC']:.4f} | "
          f"TP={row['TP']}  FN={row['FN']}  FP={row['FP']}")

final_df = pd.DataFrame(final_rows)
print("\n=== FINAL LEADERBOARD (sorted by Recall) ===")
display(style_results(final_df, sort_by="Recall"))

Balanced RF (base)                  | thresh=0.618 | Recall=0.3095  Prec=0.1232  F1=0.1763  AUC=0.7523 | TP=26  FN=58  FP=185
XGBoost_weighted (base)             | thresh=0.488 | Recall=0.4524  Prec=0.1019  F1=0.1663  AUC=0.7703 | TP=38  FN=46  FP=335
Balanced RF (tuned)                 | thresh=0.559 | Recall=0.2738  Prec=0.1204  F1=0.1673  AUC=0.7618 | TP=23  FN=61  FP=168
XGBoost (tuned)                     | thresh=0.354 | Recall=0.5119  Prec=0.0867  F1=0.1483  AUC=0.7639 | TP=43  FN=41  FP=453
Stacking (BRF + XGB)                | thresh=0.691 | Recall=0.5119  Prec=0.1057  F1=0.1752  AUC=0.7848 | TP=43  FN=41  FP=364

=== FINAL LEADERBOARD (sorted by Recall) ===


,Model,Threshold,Precision,Recall,F1,ROC-AUC,TP,TN,FP,FN
3,XGBoost (tuned),0.3539,0.0867,0.5119,0.1483,0.7639,43,"4,489",453,41
4,Stacking (BRF + XGB),0.6915,0.1057,0.5119,0.1752,0.7848,43,"4,578",364,41
1,XGBoost_weighted (base),0.4878,0.1019,0.4524,0.1663,0.7703,38,"4,607",335,46
0,Balanced RF (base),0.6180,0.1232,0.3095,0.1763,0.7523,26,"4,757",185,58
2,Balanced RF (tuned),0.5592,0.1204,0.2738,0.1673,0.7618,23,"4,774",168,61


## Step 13 — Improvement Summary

In [14]:
# Compare v1 best (Balanced RF @ threshold 0.5) vs v2 best
v1_row = baseline_df[baseline_df["Model"] == "Balanced RF (base)"].iloc[0]
v2_best = final_df.sort_values("Recall", ascending=False).iloc[0]

print("=" * 55)
print("IMPROVEMENT SUMMARY: v1 Baseline → v2 Best Model")
print("=" * 55)
print(f"{'Metric':<15} {'v1 (BRF @ 0.5)':>18} {'v2 Best':>18} {'Delta':>10}")
print("-" * 55)
for metric in ["Recall", "Precision", "F1", "ROC-AUC", "TP", "FN", "FP"]:
    v1_val = v1_row[metric]
    v2_val = v2_best[metric]
    delta  = v2_val - v1_val
    arrow  = "▲" if delta > 0 else "▼"
    fmt    = "{:.4f}" if metric not in ["TP","FN","FP"] else "{:,}"
    print(f"{metric:<15} {fmt.format(v1_val):>18} {fmt.format(v2_val):>18} "
          f"{arrow} {abs(delta):.4f}")

print("=" * 55)
print(f"Best v2 model: {v2_best['Model']} @ threshold={v2_best['Threshold']:.3f}")

IMPROVEMENT SUMMARY: v1 Baseline → v2 Best Model
Metric              v1 (BRF @ 0.5)            v2 Best      Delta
-------------------------------------------------------
Recall                      0.4762             0.5119 ▲ 0.0357
Precision                   0.0594             0.0867 ▲ 0.0273
F1                          0.1057             0.1483 ▲ 0.0426
ROC-AUC                     0.7523             0.7639 ▲ 0.0116
TP                              40                 43 ▲ 3.0000
FN                              44                 41 ▼ 3.0000
FP                             633                453 ▼ 180.0000
Best v2 model: XGBoost (tuned) @ threshold=0.354


In [16]:
import joblib

# Save the best model (Stacking ensemble — highest AUC + best FP/recall tradeoff)
best_model = stacking_pipeline
best_threshold = best_thresholds["Stacking (BRF + XGB)"]

joblib.dump(
    {"model": best_model, "threshold": best_threshold},
    "../exports/best_model.joblib"
)

print(f"Model saved → best_model.joblib")
print(f"Threshold  → {best_threshold:.4f}")

Model saved → best_model.joblib
Threshold  → 0.6915
